# 01. Record once: exposure, upgrade-targeted phrasings, activation, and gate G1

**Design decisions.**
- Every model is served by vLLM through its OpenAI-compatible endpoint, one model on the GPU at a time, greedy decoding (temperature 0, seed 0), `--generation-config vllm` so that no model-specific sampling defaults apply. Qwen3 runs with thinking off.
- Episodes are keyed by what the reader sees (model, query, memories in context, canary, temperature, seed) and written under `reports/01_records/<model>/`. A rerun after a disconnect resumes; a session whose episodes are all recorded does not start a server; records made by an older harness (`src/config.RUNTIME_VERSION`) are re-run automatically.
- Every session first runs 40 clean tasks and stops if more than 30% end on a parse failure, so a protocol problem shows before the long runs.
- A4 (plan v1, Section 7.3): the eight phrasings of each A4 record are fixed in `suite_data/` (deterministic templates), not paraphrases written by the old reader. A phrasing is eligible when the reference retriever returns it for at least one query of its category, the old reader of the record's group never acts on it (2 probes) and the group's surrogate of the new reader does (1 probe). A4 keeps the first eligible phrasing; A5 keeps the eligible phrasing with the lowest scanner score. The real new readers are never consulted, so A4 and A5 are transfer attacks.
- Activation (Equation 2): 2 probes per unit, contexts built with the new retriever R1 of each migration: bge-large-en-v1.5 for M1 to M4 and M6 (all three readers), bge-base-en-v1.5 for M5 (Qwen2.5-7B). Stores: `main` (A4 phrasings) for every run, `a5` (A5 phrasings) for the bge-large-en-v1.5 runs; probes identical to `main` are not re-run.
- Gate G1 (plan v1, Section 8) is computed on A1 to A3 units of all 100 users that the reference retriever exposes. A4 and A5 are reported separately, because the plan's ladder treats A4 as designed to flip.
- Light differential probes, a score feature, are recorded in notebook 02 with the other score features; they are not inputs to the gate.
- Gated models: Llama-3.1-8B-Instruct and Llama-3.2-3B-Instruct need a Hugging Face token with their licences accepted. The scanner is Prompt Guard 2 (86M) when its licence is accepted, otherwise ProtectAI deberta-v3-base-prompt-injection-v2; the one used is written to the reports.

In [ ]:
# === SETTINGS ===
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
VLLM_PORT       = 8000
VLLM_GPU_FRAC   = 0.90
MAX_MODEL_LEN   = 8192
VLLM_EXTRA_ARGS = ["--generation-config", "vllm"]
WORKERS         = 48            # concurrent episodes sent to the server

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import os, sys, json, shutil, subprocess, importlib, time
from pathlib import Path
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/MEMSHIFT_Research")
REPO_DIR   = DRIVE_ROOT / "memshift-research"
CRED       = DRIVE_ROOT / ".secrets"
for name, dot in [("git-credentials", ".git-credentials"), ("gitconfig", ".gitconfig")]:
    if (CRED / name).exists():
        shutil.copy(CRED / name, Path.home() / dot)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"])
os.environ["MEMSHIFT_DRIVE_ROOT"] = str(DRIVE_ROOT)
os.environ["MEMSHIFT_REPO_ROOT"] = str(REPO_DIR)
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()
pull = subprocess.run(["git", "pull", "origin", "main"], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())
print("repo:", REPO_DIR)

## Stage 1. Install vLLM and the retrieval models' library; Hugging Face login

In [ ]:
# === STAGE 1: install, login ===
import subprocess, sys
def pip(*a):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], capture_output=True, text=True)
    print("pip", *a, "-> rc", r.returncode); print(r.stderr[-800:])
    return r.returncode
_ = pip("vllm"); _ = pip("sentence-transformers", "openai", "pytest")
import importlib.metadata as md
print("vllm", md.version("vllm"), "| sentence-transformers", md.version("sentence-transformers"), "| transformers", md.version("transformers"))
from huggingface_hub import login, whoami
try:
    print("Hugging Face user:", whoami()["name"])
except Exception:
    import getpass
    login(token=getpass.getpass("Hugging Face token with the Llama 3.1 and 3.2 licences accepted: "))
    print("Hugging Face user:", whoami()["name"])

## Stage 2. Suite and repository check

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 2: load the service, run the CPU tests ===
import json
import config, suite as S
for m in (config, S): importlib.reload(m)
config.ensure_dirs()
st = S.load(config.SUITE_DATA)
print({k: len(v) for k, v in st.items()}, "| runtime version", config.RUNTIME_VERSION)
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "-p", "no:cacheprovider", "-W", "ignore"],
                   capture_output=True, text=True, cwd=REPO_DIR)
print(r.stdout[-600:]); print(r.stderr[-300:])
assert r.returncode == 0

## Stage 3. Embeddings for the four retrievers and scanner scores (GPU, before any vLLM server)

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 3: embedding caches under data/embeddings/, reports/01_scanner_scores.csv ===
import memory as M, probes as P, attacks as AT, analysis as AN, pipeline as PL, serving as SV
for m in (M, P, AT, AN, SV, PL): importlib.reload(m)
t0 = time.time()
caches = PL.embed_everything(st)
scores, scanner_name = PL.scanner_scores(st)
base = M.store_texts(st)
rets_base = PL.retrievals(st, caches, base)
ret_ref, cache_ref = rets_base[config.REF_RETRIEVER], caches[config.REF_RETRIEVER]
retrieved = PL.retrieved_map(st, ret_ref, cache_ref)
clean_texts = M.store_texts(st, include_poison=False)
ret_clean = M.Retrieval(st, clean_texts, cache_ref)
clean_q = P.clean_sample(st, config.N_CLEAN_TASKS)
expo0 = PL.exposure_table(st, {"base": rets_base})
pois = expo0[expo0.record_id.map(lambda r: st["records"][r]["poison"])]
display(pois.groupby("retriever").agg(exposed_units=("count", lambda c: int((c > 0).sum())), retrievals=("count", "sum")))
print(f"scanner: {scanner_name} | A4 phrasings retrieved by the reference retriever: {sum(retrieved.values())}/{len(retrieved)}"
      f" | {time.time() - t0:.0f} s")

## Stage 4. Phrasing checks for A4: surrogates, then the old readers

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 4: one session per model; Qwen2.5-7B stays up for Stage 6 ===
server = SV.Server(port=VLLM_PORT, gpu_frac=VLLM_GPU_FRAC, max_model_len=MAX_MODEL_LEN, extra_args=VLLM_EXTRA_ARGS)
smoke_for = lambda tag: P.clean_specs(st, ret_clean, clean_texts, clean_q[:config.N_SMOKE_TASKS], tag)
sessions = PL.Sessions(st, server, smoke_for, workers=WORKERS)
var_eps = {}
for tag in PL.VARIANT_ORDER:
    t0 = time.time()
    specs = PL.variant_specs_for(tag, st, ret_ref, cache_ref, base, retrieved)
    var_eps[tag] = sessions.run(tag, [("variant", specs)], keep_up=(tag == PL.ACTIVATION_ORDER[0]))["variant"]
    hits = sum(e["attack_hit"] for e in var_eps[tag])
    print(f"{tag}: {len(var_eps[tag])} phrasing probes, attacker outcome in {hits} | {(time.time() - t0) / 60:.1f} min")

## Stage 5. A4 and A5 selection, final stores, exposure tables

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 5: reports/01_a4_selection.csv, reports/01_exposure.csv.gz ===
sel = AT.select(st, {t: AT.variant_outcomes(e) for t, e in var_eps.items()}, scores, retrieved)
sel.to_csv(config.REPORTS / "01_a4_selection.csv", index=False)
display(sel.groupby("group").agg(records=("record_id", "size"), with_eligible_phrasing=("a4_eligible", "mean"),
                                 mean_eligible=("n_eligible", "mean"), mean_inert_old=("n_inert_old", "mean"),
                                 mean_active_surrogate=("n_active_surrogate", "mean"),
                                 a4_score=("a4_score", "mean"), a5_score=("a5_score", "mean")).round(3))
texts = {"main": M.store_texts(st, dict(zip(sel.record_id, sel.a4_variant))),
         "a5": M.store_texts(st, dict(zip(sel.record_id, sel.a5_variant)))}
rets = {k: PL.retrievals(st, caches, v) for k, v in texts.items()}
units = {k: PL.probe_units(st, r) for k, r in rets.items()}
expo = PL.exposure_table(st, rets)
expo.to_csv(config.REPORTS / "01_exposure.csv.gz", index=False)
ref_counts = rets["main"][config.REF_RETRIEVER].exposure_counts(st)
stab_units = PL.stability_units([u for u in units["main"] if ref_counts.get(u, 0) > 0])
print({k: len(v) for k, v in units.items()}, "probe units | stability units:", len(stab_units))

## Stage 6. Activation, clean tasks and stability: Qwen2.5-7B, Qwen3-8B, Llama-3.1-8B

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 6: activation probes (Equation 2) per reader and context retriever ===
act_eps, clean_eps, stab_eps = [], {}, {}
for tag in PL.ACTIVATION_ORDER:
    t0 = time.time()
    groups = []
    for reader, ctx in config.ACTIVATION_RUNS:
        if reader == tag:
            for store in PL.ACTIVATION_STORES[ctx]:
                groups.append((f"act_{store}_{ctx}", P.activation_specs(st, rets[store][ctx], texts[store], units[store], tag, ctx, store)))
    groups.append(("clean", P.clean_specs(st, ret_clean, clean_texts, clean_q, tag)))
    groups.append(("stability", P.stability_specs(st, rets["main"][config.REF_RETRIEVER], texts["main"], stab_units, tag)))
    out = sessions.run(tag, groups)
    for k, v in out.items():
        if k.startswith("act_"):
            act_eps += v
    clean_eps[tag], stab_eps[tag] = out["clean"], out["stability"]
    print(f"{tag}: " + ", ".join(f"{k} {len(v)}" for k, v in out.items()) + f" | {(time.time() - t0) / 60:.1f} min")
server.stop()

## Stage 7. Flips, Equation 4, retrieval flips, gate G1

In [ ]:
import sys, importlib
SRC = str(REPO_DIR / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()          # src/ may have been written after the bootstrap; the Drive mount does not refresh the importer cache
assert (REPO_DIR / "src" / "config.py").exists(), "src/config.py missing: run notebook 00 first"

# === STAGE 7: reports/01_activation.csv, 01_flips_*.csv, 01_flip_summary_*.csv, 01_decomposition.csv, 01_claims.json ===
act = AN.activation_frame(act_eps)
act.to_csv(config.REPORTS / "01_activation.csv", index=False)
display(act.groupby(["store", "reader", "ctx_retriever"]).agg(units=("a", "size"), active_share=("a", "mean"),
        parse_failures=("parse_failures", "sum")).round(3))
pops = {"all": set(st["users"]), "test": {u for u, d in st["users"].items() if d["split"] == "test"}}
summ, decs, rflips = {}, [], []
for store in ("main", "a5"):
    g = AN.Ground(st, expo, act, store, pops["all"])
    migs = list(config.MIGRATIONS) if store == "main" else config.READER_MIGRATIONS
    fl = pd.concat([AN.flips(g, m) for m in migs], ignore_index=True)
    fl.to_csv(config.REPORTS / f"01_flips_{store}.csv", index=False)
    s_att = AN.flip_summary(fl, ["migration", "attack"])
    s_att.to_csv(config.REPORTS / f"01_flip_summary_{store}.csv", index=False)
    AN.flip_summary(fl, ["migration", "attack", "category"]).to_csv(config.REPORTS / f"01_flip_summary_{store}_by_category.csv", index=False)
    summ[store] = (fl, s_att)
    if store == "main":
        a4 = fl[fl.attack == "A4"].copy(); a4["own_group"] = a4.group == a4.migration
        AN.flip_summary(a4, ["migration", "own_group"]).to_csv(config.REPORTS / "01_flip_summary_main_a4_groups.csv", index=False)
        for pop, users in pops.items():
            gp = AN.Ground(st, expo, act, store, users)
            decs += [{"population": pop, **AN.decomposition(gp, m)} for m in config.MIGRATIONS]
        rflips = [AN.retrieval_flips(g, m) for m in ("M4", "M5", "M6")]
print("Newly active share among units inert under the old reader (main store):")
display(summ["main"][1].round(3))
dec = pd.DataFrame(decs); dec.to_csv(config.REPORTS / "01_decomposition.csv", index=False)
print("Equation 4 (active share of memory retrievals):"); display(dec.round(5))
rf = pd.DataFrame(rflips); rf.to_csv(config.REPORTS / "01_retrieval_flips.csv", index=False); display(rf.round(5))
print("A5 store, A4 records only:"); display(summ["a5"][1][summ["a5"][1].attack == "A4"].round(3))
cu = PL.clean_utility(clean_eps); cu.to_csv(config.REPORTS / "01_clean_utility.csv", index=False); display(cu.round(3))
stab = PL.stability_table(stab_eps, act); stab.to_csv(config.REPORTS / "01_stability.csv", index=False); display(stab.round(3))
gate = AN.gate_g1(summ["main"][0])
claims = {"gate_G1": gate, "scanner": scanner_name, "a4_records_with_eligible_phrasing": int(sel.a4_eligible.sum()),
          "a4_records": int(len(sel)), "probe_units": {k: len(v) for k, v in units.items()},
          "decomposition_all_users": dec[dec.population == "all"].set_index("migration").drop(columns="population").round(6).to_dict("index"),
          "runtime_version": config.RUNTIME_VERSION}
(config.REPORTS / "01_claims.json").write_text(json.dumps(claims, indent=2))
print(json.dumps(gate, indent=2))

## End of unit: commit

Save the notebook first (File > Save), then run the cell below.

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py",
    'nb01: ground truth recorded once: embeddings for four retrievers, scanner scores (reports/01_scanner_scores.csv), A4 phrasing checks on surrogates and old readers and the A4/A5 selection (reports/01_a4_selection.csv), exposure tables (reports/01_exposure.csv.gz), activation probes for Qwen2.5-7B, Qwen3-8B and Llama-3.1-8B (reports/01_records/), clean tasks, stability, flips, Equation 4 decomposition, retrieval flips and the gate G1 verdict (reports/01_activation.csv, 01_flips_*.csv, 01_decomposition.csv, 01_claims.json)'],
    capture_output=True, text=True)
print(r.stdout); print(r.stderr)